# 04 · Code graph — a codebase as queryable memory

`run_code_graph_pipeline` parsed a Python package into a graph of modules/classes/functions in AgensGraph (`cognee_code`). Query it with `SearchType.CODE` and `INSIGHTS`, and visualize it.

> Run `build.py` first.

In [1]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))  # examples/demos
from _common import config
config.require_openai_key(); config.quiet()          # quiet cognee's verbose logs
config.configure("cognee_code")
import cognee
from cognee.modules.search.types import SearchType
from cognee.infrastructure.databases.graph import get_graph_engine

def _name(node):
    if isinstance(node, dict):
        return str(node.get("name") or (node.get("text") or "")[:40] or node.get("id") or "?")
    return str(node)[:40]

def render(results):
    if isinstance(results, (str, bytes)) or not isinstance(results, (list, tuple)):
        results = [results] if results else []
    for r in results[:4]:
        if isinstance(r, (tuple, list)) and len(r) == 3:
            src, edge, tgt = r
            rel = edge.get("relationship_name") if isinstance(edge, dict) else str(edge)
            print(f"   ({_name(src)}) -[{rel}]-> ({_name(tgt)})")
        elif isinstance(r, dict):
            print("   " + str(r.get("text") or r.get("name") or r)[:150])
        else:
            print("   " + str(r).strip().replace(chr(10), " ")[:550])
m = await (await get_graph_engine()).get_graph_metrics(include_optional=False)
print("code graph:", m["num_nodes"], "nodes,", m["num_edges"], "edges")

code graph: 261 nodes, 389 edges


## Semantic code search + the extracted code structure

In [2]:
from collections import Counter
q = "How does the library send an HTTP request?"
print("CODE search:")
for r in (await config.search(query_text=q, query_type=SearchType.CODE))[:3]:
    if isinstance(r, dict):
        print(f"   {str(r.get('name','?')).split('/')[-1]}: {' '.join((r.get('content') or '').split())[:90]}")
print("\ncode graph structure:")
nodes, _ = await (await get_graph_engine()).get_graph_data()
for t, c in Counter(p.get("type") for _, p in nodes).most_common():
    print(f"   {t}: {c}")

CODE search:


   status_codes.py: r""" The ``codes`` object defines a mapping from common names for HTTP statuses to their n
   __init__.py: # __ # /__) _ _ _ _ _/ _ # / ( (- (/ (/ (- _) / _) # / """ Requests HTTP Library ~~~~~~~~~
   utils.py: """ requests.utils ~~~~~~~~~~~~~~ This module provides utility functions that are used wit

code graph structure:
   ImportStatement: 125
   FunctionDefinition: 72
   ClassDefinition: 44
   CodeFile: 19
   Repository: 1


## Visualize the code graph

In [3]:
out = str(config.DATA_DIR / "code_graph.html")
await cognee.visualize_graph(out)
print("wrote", out)

wrote .data/code_graph.html
